### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a straightforward biological/behavioral question about parrot vocalization and speech mimicry.\n\n2.  **Identify Key Concepts**:\n   - Parrots don\'t actually "talk" in the human sense (they don\'t understand language fully)\n   - They mimic sounds/vocalizations\n   - Evolutionary/social reasons for mimicry\n   - Biological mechanisms (syrinx, brain structure)\n   - Social bonding, communication within flock\n   - Environmental adaptation\n   - Domestication/captive environment influence\n\n3.  **Structure the Answer**:\n   - Clarify what "talking" really means (mimicry vs. language)\n   - Explain the biological/physical mechanism\n   - Discuss evolutionary/social reasons\n   - Address captive vs. wild behavior\n   - Note cognitive aspects (some understanding, but not full language)\n   - Keep it clear, scientific, and accessible\n\n4.  **Draft - S

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])


In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in Boston.\n2.  Identify the required tool: `get_weather`\n3.  Identify the required parameters: `location` = "Boston"\n4.  Execute the tool call.\n5.  Format the response based on the tool\'s output.\n\nLet\'s call the tool.✅\n', 'tool_calls': [{'id': 'ysx2pae3d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 275, 'total_tokens': 387, 'completion_time': 0.213059569, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.019458781, 'prompt_tokens_details': None, 'queue_time': 0.052126139, 'total_time': 0.23251835}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f8

In [11]:
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston.


In [13]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the weather in Boston.\n2.  Identify available tools: `get_weather(location: string)`\n3.  Extract parameters: location = "Boston"\n4.  Call tool: `get_weather(location="Boston")`\n5.  Format response based on tool output. (I will simulate the tool call) -> Wait, I need to actually call the tool.\n6.  Execute tool call.🌟\n', 'tool_calls': [{'id': '49ck9d0q0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 274, 'total_tokens': 406, 'completion_time': 0.290715401, 'completion_tokens_details': {'reasoning_tokens': 104}, 'prompt_time': 0.02147999, 'prompt_tokens_details': None, 'queue_time': 0.045822099, 'total_time': 0.312195391}, 'model_name': 'qwen/qwen3.6-27b', 'system_fi